# Weather data cleaning
This notebook documents the cleaning and preprocessing of hourly weather data for Dubrovnik (2024-2025), preparing the dataset for integration with Uber trip records and further analysis.

## Data loading and dataset overview
The initial inspection is performed to understand the dataset structure, data types and potential missing values

In [7]:
import pandas as pd
import numpy as np

In [8]:
df = pd.read_csv('../../data/raw_data/weather_hourly_2024_2025.csv')

In [9]:
df.head()

,time,temperature_2m,precipitation,weathercode,apparent_temperature
0,2024-04-19T00:00,8.5,0.0,1,6.2
1,2024-04-19T01:00,8.9,0.0,3,6.4
2,2024-04-19T02:00,9.2,0.0,3,5.8
3,2024-04-19T03:00,8.3,0.0,3,4.9
4,2024-04-19T04:00,8.2,0.0,3,4.7


In [10]:
df.tail()

,time,temperature_2m,precipitation,weathercode,apparent_temperature
11947,2025-08-29T19:00,24.8,0.3,51,24.7
11948,2025-08-29T20:00,24.4,0.1,51,24.4
11949,2025-08-29T21:00,23.4,0.2,51,24.2
11950,2025-08-29T22:00,23.3,0.0,2,23.5
11951,2025-08-29T23:00,22.5,2.5,63,24.2


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11952 entries, 0 to 11951
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  11952 non-null  object 
 1   temperature_2m        11952 non-null  float64
 2   precipitation         11952 non-null  float64
 3   weathercode           11952 non-null  int64  
 4   apparent_temperature  11952 non-null  float64
dtypes: float64(3), int64(1), object(1)
memory usage: 467.0+ KB


In [12]:
df['time'] = pd.to_datetime(df['time'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11952 entries, 0 to 11951
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time                  11952 non-null  datetime64[ns]
 1   temperature_2m        11952 non-null  float64       
 2   precipitation         11952 non-null  float64       
 3   weathercode           11952 non-null  int64         
 4   apparent_temperature  11952 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1)
memory usage: 467.0 KB


In [13]:
df.isnull().sum()

time                    0
temperature_2m          0
precipitation           0
weathercode             0
apparent_temperature    0
dtype: int64

In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
df['time'].duplicated().sum()

np.int64(0)

In [17]:
expected_hours = pd.date_range(
    start = df['time'].min(),
    end = df['time'].max(),
    freq = 'h'
)
missing_hours = expected_hours.difference(df['time'])
print('Missing hours:', len(missing_hours))

Missing hours: 0


In [18]:
df['time'] = (
    df['time']
    .dt.tz_localize('UTC')
    .dt.tz_convert('Europe/Zagreb')
)

In [20]:
df[['time']].head()

,time
0,2024-04-19 02:00:00+02:00
1,2024-04-19 03:00:00+02:00
2,2024-04-19 04:00:00+02:00
3,2024-04-19 05:00:00+02:00
4,2024-04-19 06:00:00+02:00


Converted weather timestamps from UTC to Europe/Zagreb to align with Uber trip timestamps.

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11952 entries, 0 to 11951
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype                        
---  ------                --------------  -----                        
 0   time                  11952 non-null  datetime64[ns, Europe/Zagreb]
 1   temperature_2m        11952 non-null  float64                      
 2   precipitation         11952 non-null  float64                      
 3   weathercode           11952 non-null  int64                        
 4   apparent_temperature  11952 non-null  float64                      
dtypes: datetime64[ns, Europe/Zagreb](1), float64(3), int64(1)
memory usage: 467.0 KB


In [22]:
df.describe()

,temperature_2m,precipitation,weathercode,apparent_temperature
count,11952.000000,11952.000000,11952.000000,11952.000000
mean,17.904903,0.202276,9.560074,17.455622
std,7.897642,0.910997,19.865827,9.877012
min,-1.500000,0.000000,0.000000,-6.200000
25%,12.000000,0.000000,0.000000,9.975000
50%,17.900000,0.000000,1.000000,17.700000
75%,23.800000,0.000000,3.000000,24.900000
max,36.800000,17.500000,73.000000,42.600000


### Filtering weather data by Uber driving seasons

In [23]:
df = df[
    ((df['time'] >= '2024-04-19') & (df['time'] < '2024-10-04')) |
    ((df['time'] >= '2025-07-11') & (df['time'] < '2025-08-30'))
    ].copy()
    

In [24]:
df.shape

(5230, 5)

In [25]:
df['time'].dt.year.value_counts().sort_index()

time
2024    4030
2025    1200
Name: count, dtype: int64

In [26]:
df.reset_index(drop=True, inplace=True)

In [27]:
df.head()

,time,temperature_2m,precipitation,weathercode,apparent_temperature
0,2024-04-19 02:00:00+02:00,8.5,0.0,1,6.2
1,2024-04-19 03:00:00+02:00,8.9,0.0,3,6.4
2,2024-04-19 04:00:00+02:00,9.2,0.0,3,5.8
3,2024-04-19 05:00:00+02:00,8.3,0.0,3,4.9
4,2024-04-19 06:00:00+02:00,8.2,0.0,3,4.7


After reviewing the weather data, I decided to focus on rainfall rather than temperature and other weather conditions, as the main objective is to compare Uber earnings during rainy and dry periods.

In [28]:
df = df[['time', 'precipitation']].copy()
df['is_raining'] = df['precipitation'] > 0

In [29]:
df.head()

,time,precipitation,is_raining
0,2024-04-19 02:00:00+02:00,0.0,False
1,2024-04-19 03:00:00+02:00,0.0,False
2,2024-04-19 04:00:00+02:00,0.0,False
3,2024-04-19 05:00:00+02:00,0.0,False
4,2024-04-19 06:00:00+02:00,0.0,False


In [31]:
df.to_csv('../../data/processed_data/weather_hourly_cleaned.csv' , index = False)